## Получение списка дат экспираций

# Тестирование сохранения данных опционов в БД

Этот notebook используется для проверки работы периодического сохранения данных из WebSocket в SQLite базу данных.

## Инструкция по тестированию:

1. **Запустите приложение в Docker:**
   ```bash
   docker compose up --build
   ```

2. **Дождитесь подключения WebSocket и начала сбора данных:**
   - В логах должно появиться сообщение "✅ WebSocket connected"
   - Должно появиться "Периодическое сохранение данных в БД запущено"
   - Подождите минимум 5-10 минут, чтобы накопились данные

3. **Если БД находится в Docker контейнере, скопируйте её локально:**
   ```bash
   docker cp option-telegram-bot:/app/data/options.db ./data/options.db
   ```
   Или если БД монтируется как volume, она уже будет доступна локально.

4. **Запустите ячейки ниже для проверки данных**

## Строка подключения к БД:

**Локально:** `./data/options.db` (относительно корня проекта)
**В Docker:** `/app/data/options.db` (внутри контейнера)

In [ ]:
from pybit.unified_trading import HTTP

def get_option_expirations_pybit():
    session = HTTP(testnet=False)
    
    try:
        response = session.get_instruments_info(
            category="option",
            status="Trading",
            baseCoin="BTC",
            settleCoin="USDT",
            limit=1000
        )
        
        if response['retCode'] == 0:
            expiry_dates = set()
            for instrument in response['result']['list']:
                if instrument['settleCoin'] == 'USDT':
                    expiry_dates.add(instrument['deliveryTime'])
            
            return sorted(list(expiry_dates))
        else:
            print(f"Ошибка: {response['retMsg']}")
            return []
    except Exception as e:
        print(f"Исключение: {e}")
        return []

In [ ]:
get_option_expirations_pybit()

## Тестирование

In [ ]:
import sys

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import display

# Путь к базе данных
# В Docker контейнере база находится в /app/data/options.db
# Локально - в ./data/options.db относительно корня проекта
project_root = Path.cwd()
db_path = project_root / "data" / "options.db"

print(f"Путь к БД: {db_path}")
print(f"БД существует: {db_path.exists()}")

# Строка подключения для SQLite (просто путь к файлу)
connection_string = str(db_path)
print(f"\nСтрока подключения: {connection_string}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Подключение к БД
conn = sqlite3.connect(str(db_path))
conn.row_factory = sqlite3.Row  # Для доступа к колонкам по имени

# Проверка таблиц
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Таблицы в БД:")
for table in tables:
    print(f"  - {table[0]}")

In [ ]:
# 1. Проверка количества записей в option_history
query = "SELECT COUNT(*) as count FROM option_history"
df = pd.read_sql_query(query, conn)
print("Количество записей в option_history:")
print(df)

In [ ]:
# 2. Последние 10 записей из option_history
query = """
SELECT 
  *
FROM option_history
"""
df = pd.read_sql_query(query, conn)
print("Последние 10 записей:")
display(df.head())

In [ ]:
df['iv'].describe()

In [ ]:
query = """
SELECT 
    *
FROM option_history
ORDER BY timestamp DESC
LIMIT 3
"""
df = pd.read_sql_query(query, conn)
print("Последние 10 записей:")
print(df.to_string())

In [ ]:
# 3. Уникальные символы в БД
query = """
SELECT DISTINCT symbol, COUNT(*) as count
FROM option_history
GROUP BY symbol
ORDER BY count DESC
"""
df = pd.read_sql_query(query, conn)
print("Уникальные символы и количество записей:")
print(df.to_string())

In [ ]:
# 4. Проверка временных интервалов сохранения (должны быть кратны 5 минутам)
query = """
SELECT 
    timestamp,
    strftime('%M', timestamp) as minute,
    COUNT(*) as count
FROM option_history
GROUP BY timestamp
ORDER BY timestamp DESC
LIMIT 20
"""
df = pd.read_sql_query(query, conn)
print("Временные метки сохранения (последние 20):")
print(df.to_string())

# Проверяем, что минуты кратны 5
if len(df) > 0:
    minutes = df['minute'].astype(int)
    not_aligned = minutes[minutes % 5 != 0]
    if len(not_aligned) > 0:
        print(f"\n⚠️ ВНИМАНИЕ: Найдены записи не выровненные по 5 минутам: {not_aligned.tolist()}")
    else:
        print("\n✅ Все временные метки выровнены по 5-минутным интервалам")

In [ ]:
# 5. История для конкретного символа (если есть данные)
query = """
SELECT symbol
FROM option_history
LIMIT 1
"""
df_symbol = pd.read_sql_query(query, conn)

if len(df_symbol) > 0:
    test_symbol = df_symbol['symbol'].iloc[0]
    print(f"Анализ истории для символа: {test_symbol}\n")
    
    query_history = """
    SELECT 
        timestamp,
        ask_price,
        bid_price,
        mark_price,
        iv,
        delta,
        underlying_price
    FROM option_history
    WHERE symbol = ?
    ORDER BY timestamp ASC
    """
    df_history = pd.read_sql_query(query_history, conn, params=(test_symbol,))
    print(f"Количество записей: {len(df_history)}")
    print("\nПервые 5 записей:")
    print(df_history.head().to_string())
    print("\nПоследние 5 записей:")
    print(df_history.tail().to_string())
else:
    print("В БД пока нет данных")

In [ ]:
# 6. Статистика по underlying_history
query = """
SELECT 
    symbol,
    COUNT(*) as count,
    MIN(timestamp) as first_record,
    MAX(timestamp) as last_record
FROM underlying_history
GROUP BY symbol
"""
df = pd.read_sql_query(query, conn)
print("Статистика по базовым активам:")
print(df.to_string())

In [ ]:
# 7. Проверка IV истории
query = """
SELECT 
    symbol,
    COUNT(*) as count,
    AVG(iv) as avg_iv,
    MIN(iv) as min_iv,
    MAX(iv) as max_iv
FROM iv_history
WHERE iv IS NOT NULL
GROUP BY symbol
ORDER BY count DESC
LIMIT 10
"""
df = pd.read_sql_query(query, conn)
print("Статистика IV по символам:")
print(df.to_string())

In [ ]:
# Закрываем соединение
conn.close()
print("Соединение с БД закрыто")